In [1]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

# 1. Loading the document

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [3]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

# 2. Spliting document into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [6]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [7]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [8]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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"
)

# 6. Building Prompts and chains

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [11]:
def ask(query):
    top_docs = faiss_db.similarity_search(query, k=10)
    prompt_template = PromptTemplate.from_template(
      "Give answer according to the following passages in the quran {context}"
      "If the answer is not present in the given context then give answer according to the internet sources."
      "But do inform that there are no passages in the quran about the question."
      "Answer the following question {question}."
    )
    chain = prompt_template | llm | StrOutputParser()
    result = chain.invoke({"context": top_docs, "question": query})
    return result

Relevant questions

In [12]:
asnwer = ask("What does the Quran say about Day of Judgment?")
print(asnwer)

The Quran provides extensive information about the Day of Judgment, also known as the Day of Reckoning or Qiyamah. Here are some key points mentioned in the Quran:

1. **The Day of Judgment is inevitable**: The Quran emphasizes that the Day of Judgment is a certainty and will inevitably occur (Surah 3:185, Surah 6:73, Surah 22:1).
2. **All souls will be held accountable**: On the Day of Judgment, every individual will be held accountable for their deeds, and their souls will be judged based on their actions (Surah 82:19, Surah 21:47, Surah 54:52).
3. **The Book of Deeds will be presented**: The Quran mentions that a Book of Deeds will be presented to each individual, which will contain a record of all their actions, good and bad (Surah 17:71, Surah 18:49, Surah 54:52).
4. **The weighing of deeds**: The Quran mentions that the deeds of each individual will be weighed on the Day of Judgment, and those whose good deeds outweigh their bad deeds will be rewarded (Surah 7:8, Surah 21:47, Sur

In [17]:
asnwer = ask("What is the importance of RAMADAN is islam?")
print(asnwer)

The importance of Ramadan in Islam can be understood from the given passages, although they do not directly answer the question. However, from the passages, we can infer the significance of Ramadan as a holy month.

In Surah 44 (Ad-Dukhan), it is mentioned that every year has a particular identity and importance in the total history of the universe, and there is a special night during each year in which every thing or being that God has willed to come into existence, and every event that has been willed to take place during that year, is identified or particularized and transferred from Divine Knowledge to the disposal of the Divine Power. This night is the Night of Destiny (or Power and Measure), which occurs in the Holy Month of Ramadan.

In Surah 97 (Al-Qadr), it is mentioned that the Night of Destiny and Power is better than a thousand months, and the angels and the Spirit descend in it by the permission of their Lord with His decrees for every affair. This night is a time of great

Irrelevant questions

In [14]:
asnwer = ask("When did dinosaurs came into being?")
print(asnwer)

There is no passage in the Quran that directly mentions dinosaurs or their existence. The Quran does mention the creation of animals and living beings, but it does not provide specific information about dinosaurs.

According to internet sources and scientific research, dinosaurs are believed to have roamed the Earth during the Mesozoic Era, which lasted from about 252 million to 66 million years ago. The exact timing of their emergence is not certain, but it is thought to have occurred during the Middle to Late Triassic period, around 230-245 million years ago.

Some of the most well-known types of dinosaurs, such as the sauropods and the theropods, are believed to have evolved during the Jurassic period, around 200-150 million years ago. The dinosaurs continued to dominate the Earth's landscapes until their sudden extinction at the end of the Cretaceous period, about 66 million years ago.

Please note that the information about dinosaurs is based on scientific research and fossil reco